In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage, ToolMessage, SystemMessage
from langchain.tools import tool
from random import randint
from loadmodel import load_model
from mtools import get_weather, get_news

# 初始化模型
model = load_model()

# 绑定工具到模型
tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)

result = model_with_tools.invoke("北京的天气和科技新闻")
print(result)

content='我来为您获取北京的天气和科技新闻。\n\n' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 375, 'total_tokens': 436, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 375}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus-2026-05-26', 'system_fingerprint': None, 'id': 'chatcmpl-58f19674-9710-962b-8d12-bba24ea60bbc', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ffa0c-ed51-7c63-8f51-1047e573cba1-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_afcc84cb5a80472796f1d1a4', 'type': 'tool_call'}, {'name': 'get_news', 'args': {'domain': '科技'}, 'id': 'call_43baa1f82f0c45a288c5bcbf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 375, 'output_tokens': 61, 'total_tokens': 436, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [18]:
# 1. 定义工具
class OverAllState(MessagesState):
    user_input: str = ""
    final_output: str = ""

# 2. 声明节点
# 2.1 输入节点 => 将用户输入的查询信息 记录到message中，方便后续的大模型调用
def input_node(state:OverAllState) -> OverAllState:
    return{
        "messages": [HumanMessage(content=state["user_input"])]
    }

# 2.2 大模型节点
def llm_node(state:OverAllState) -> OverAllState:
    ai_message = model_with_tools.invoke(state["messages"])
    return{
        "messages": [ai_message]
    }

# 2.3 判断是否需要调用工具
def tool_node(state:OverAllState) -> OverAllState:
    # 判断大模型的返回信息是否需要调用tool
    messages = state["messages"]
    ai_message = messages[-1]
    tool_calls = ai_message.tool_calls
    # 模拟函数调用失败的概率
    fail_prob = 6
    for tool_call in tool_calls:
        if tool_call["name"] == "get_weather":
            # 随机生成[0-9]数字，判断大小
            rint = randint(0, 9)
            if rint < fail_prob:
                messages.append(
                    ToolMessage(
                        content="网络波动，调用失败",
                        tool_call_id=tool_call["id"]
                    )
                )
            else:
                messages.append(get_weather.invoke(tool_call))
        
        elif tool_call["name"] == "get_news":
            # 随机生成[0-9]数字，判断大小
            rint = randint(0, 9)
            if rint < fail_prob:
                messages.append(
                    ToolMessage(
                        content="网络波动，调用失败",
                        tool_call_id=tool_call["id"]
                    )
                )
            else:
                messages.append(get_news.invoke(tool_call))

        else:
            messages.append(
                ToolMessage(
                    content=f"不支持的工具调用: {tool_call['name']}",
                    tool_call_id=tool_call["id"]
                )
            )
    return{
        "messages": messages
    }

# 2.4 输出节点
def output_node(state:OverAllState) -> OverAllState:
    final_output = state["messages"][-1].content
    return{
        "final_output": final_output
    }

# 2.5 判断是否需要进行工具节点的调用，还是直接到output节点
def router(state:OverAllState) -> Literal["tool_node", "output_node"]:
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "output_node"

# 3. 构建图
builder = StateGraph(state_schema = OverAllState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_conditional_edges(
    "llm_node",
    router
)
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()

result = graph.invoke({
    "user_input": "查询今天北京的天气和科技新闻",
    "messages": [SystemMessage(content="如果工具调用失败，必须重新调用直到成功为止")]
    })
print("user_input:", result["user_input"])
print("final_output:", result["final_output"])

for message in result["messages"]:
    message.pretty_print()

# print("="*50)
# print(result)


# from IPython.display import display
# display(graph)


user_input: 查询今天北京的天气和科技新闻
final_output: 太好了！现在两个查询都成功了。让我为您总结结果：

**北京天气：**
北京的天气是晴朗的

**科技新闻：**
科技新闻内容

以上就是今天北京的天气和科技新闻的信息。
================================ System Message ================================

如果工具调用失败，必须重新调用直到成功为止
================================ Human Message =================================

查询今天北京的天气和科技新闻
================================== Ai Message ==================================

我来为您查询今天北京的天气和科技新闻。
Tool Calls:
  get_weather (call_0208f2f7f1594e6eba53c329)
 Call ID: call_0208f2f7f1594e6eba53c329
  Args:
    city: 北京
  get_news (call_36dec8623f254a4a883b6658)
 Call ID: call_36dec8623f254a4a883b6658
  Args:
    domain: 科技
================================= Tool Message =================================

网络波动，调用失败
================================= Tool Message =================================

网络波动，调用失败
================================== Ai Message ==================================

调用失败了，我需要重新尝试。让我再次查询北京的天气和科技新闻。
Tool Calls:
  get_weather (call_b603de9c83dd46b